# 第4章　データを用意する ― 公開データとアノテーションの基礎

**『医療診断支援AI開発　入門編 ― ゼロから動かす（入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-intro

## 4.4　データの置き方 ― フォルダとラベルの対応

```text
my_dataset/
├── images/          # 画像
│   ├── case_001.png
│   └── case_002.png
└── labels/          # 正解（枠やマスク、あるいはラベル表）
    ├── case_001.txt
    └── case_002.txt
```

## 手を動かす ― フォルダ構成から「対応表CSV」を自動で作る

In [ ]:
import pandas as pd
import os, glob, pandas as pd

ROOT = "wrist"     # wrist/train/normal/P01_L.png のような構成
rows = []
for split in ["train", "val"]:
    for cls in ["normal", "fracture"]:
        for path in glob.glob(f"{ROOT}/{split}/{cls}/*.png"):
            fname = os.path.basename(path)      # 例: P01_L.png
            patient = fname.split("_")[0]        # 先頭(_の前)を患者IDとみなす → "P01"
            rows.append({"path": path, "patient_id": patient,
                         "label": 0 if cls == "normal" else 1, "split": split})

df = pd.DataFrame(rows)
# ※ この対応表では label=1 が骨折（陽性）。第8章の ImageFolder は辞書順で fracture=0 に
#    なるので、両者を混ぜない（混ぜるなら pos_label をクラス名から引いて明示する）。
# 命名規約の検算。_ を含まないファイル名だと patient_id がファイル名そのものへ静かに退化する。
# ここで「患者数 < 画像数」を条件にしてはいけない。1患者1画像のデータでは両者が等しく、
# 正しいデータまで弾いてしまう。見るべきは、規約どおりの形をしているか、である。
bad = df[~df["path"].map(lambda q: "_" in os.path.splitext(os.path.basename(q))[0])]
assert bad.empty, f"命名規約（患者ID_連番）に合わないファイル: {bad['path'].tolist()[:5]}"
assert df["patient_id"].notna().all() and (df["patient_id"] != "").all(), "患者IDが空の行がある"
df.to_csv("mapping.csv", index=False)
print(df.head())
print("患者数:", df["patient_id"].nunique(), " 画像数:", len(df))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
for k, (tr, va) in enumerate(sgkf.split(df, y=df["label"], groups=df["patient_id"])):
    tr_pat, va_pat = set(df.iloc[tr]["patient_id"]), set(df.iloc[va]["patient_id"])
    leak = tr_pat & va_pat                       # 学習と検証で重なる患者
    pos  = df.iloc[va]["label"].mean()           # 検証セットの陽性率
    print(f"fold{k}: 検証 {len(va):3d}枚  陽性率 {pos:.2f}  患者リーク {len(leak)}件")
# fold0: 検証 217枚  陽性率 0.25  患者リーク 0件
# fold1: 検証 218枚  陽性率 0.26  患者リーク 0件 ...

## 置き場所には、必ず README.md を添える

```markdown
# 手関節X線データセット v2

- 出典：〇〇公開データセット（取得日 2026-05-10、ライセンス：CC BY 4.0）
- 枚数：normal 812枚 / fracture 274枚（患者数 543人）
- 分割：patient_id で 8:2（同一患者が学習と検証にまたがらないこと）
- 前処理：グレースケール（黒から白までの明るさだけで表した画像）に変換のうえ3チャネルへ複製、縦横比を保って長辺を224に縮小し余白を黒でパディング（レターボックス化）、ImageNet統計で正規化
- 除外：画質不良 12枚を excluded/ へ隔離（理由は excluded/reason.csv）
- 注意：正常画像に小児が多く含まれる。成人のみで評価する場合は要フィルタ
```